<a href="https://colab.research.google.com/github/amzad-786githumb/Privacy-Preserving-Synthetic-Tabular-Data-Generation-Using-Generative-Adversarial-Networks/blob/main/04_Baseline_Deep_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Notebook 04 : Baseline Deep Models
# Section A : Environment Setup
# Install Required Packages
# ============================================================

!pip -q install -U sdv==1.17.2
!pip -q install -U ctgan==0.10.2
!pip -q install -U copulas
!pip -q install -U torch torchvision torchaudio
!pip -q install -U scikit-learn
!pip -q install -U pandas numpy scipy matplotlib seaborn
!pip -q install -U tqdm joblib pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.0/154.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4

In [1]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import json
import yaml
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import torch

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    TVAESynthesizer
)

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
#  Reproducibility
# ============================================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():

    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random Seed :", RANDOM_STATE)

Random Seed : 42


In [3]:
# ============================================================
# GPU Configuration
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("="*70)

print("Device :", DEVICE)

if DEVICE == "cuda":

    print("GPU :", torch.cuda.get_device_name(0))

    print(
        "GPU Memory :",
        round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),
        "GB"
    )

print("="*70)

Device : cpu


In [4]:
# ============================================================
# Mount Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# ============================================================
# Project Directory
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Project"
)

print(PROJECT_ROOT)

/content/drive/MyDrive/SPP_GAN_Project


In [6]:
# ============================================================
# Directory Configuration
# ============================================================

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"

BASELINE_MODEL_DIR = MODELS_DIR / "baseline_models"

CHECKPOINT_DIR = MODELS_DIR / "checkpoints"

RESULT_DIR = PROJECT_ROOT / "results"

LOG_DIR = PROJECT_ROOT / "logs"

CONFIG_DIR = PROJECT_ROOT / "configs"

REPORT_DIR = PROJECT_ROOT / "reports"

In [7]:
# ============================================================
# Model Directories
# ============================================================

CTGAN_DIR = BASELINE_MODEL_DIR / "ctgan"

TVAE_DIR = BASELINE_MODEL_DIR / "tvae"

DP_CTGAN_DIR = BASELINE_MODEL_DIR / "dp_ctgan"

CTABGAN_DIR = BASELINE_MODEL_DIR / "ctabgan"

for folder in [

    CTGAN_DIR,

    TVAE_DIR,

    DP_CTGAN_DIR,

    CTABGAN_DIR,

    CHECKPOINT_DIR,

    RESULT_DIR,

    LOG_DIR,

    REPORT_DIR

]:

    folder.mkdir(parents=True, exist_ok=True)

print("Directories Created")

Directories Created


In [8]:
# ============================================================
# Dataset Paths
# ============================================================

DATASET_PATHS = {

    "Adult Income":

    PROCESSED_DIR / "adult_income_processed.csv",

    "Bank Marketing":

    PROCESSED_DIR / "bank_marketing_processed.csv",

    "Breast Cancer":

    PROCESSED_DIR / "breast_cancer_processed.csv"

}

In [9]:
# ============================================================
# Metadata Paths
# ============================================================

METADATA_PATHS = {

    "Adult Income":

    CONFIG_DIR / "adult_income_metadata.json",

    "Bank Marketing":

    CONFIG_DIR / "bank_marketing_metadata.json",

    "Breast Cancer":

    CONFIG_DIR / "breast_cancer_metadata.json"

}

In [15]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Project")

csv_files = list(PROJECT_ROOT.rglob("*.csv"))

print(f"Found {len(csv_files)} CSV files:\n")

for file in csv_files:
    print(file)

Found 7 CSV files:

/content/drive/MyDrive/SPP_GAN_Project/results/dataset_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/results/dataset_validation.csv
/content/drive/MyDrive/SPP_GAN_Project/results/metadata_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/results/feature_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/datasets/processed/adult_income_processed.csv
/content/drive/MyDrive/SPP_GAN_Project/datasets/processed/bank_marketing_processed.csv
/content/drive/MyDrive/SPP_GAN_Project/datasets/processed/breast_cancer_processed.csv


In [17]:
# ============================================================
# Automatically Locate Processed Datasets
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Project")

csv_files = list(PROJECT_ROOT.rglob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV datasets were found inside the project directory."
    )

print("Available CSV files:\n")

for f in csv_files:
    print(f)

Available CSV files:

/content/drive/MyDrive/SPP_GAN_Project/results/dataset_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/results/dataset_validation.csv
/content/drive/MyDrive/SPP_GAN_Project/results/metadata_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/results/feature_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/datasets/processed/adult_income_processed.csv
/content/drive/MyDrive/SPP_GAN_Project/datasets/processed/bank_marketing_processed.csv
/content/drive/MyDrive/SPP_GAN_Project/datasets/processed/breast_cancer_processed.csv


In [19]:
# ============================================================
# Verify Files
# ============================================================

# The previous cells indicated that the files are in 'datasets/processed'
# instead of 'data/processed'. Correcting PROCESSED_DIR and DATASET_PATHS here.
PROCESSED_DIR = PROJECT_ROOT / "datasets" / "processed" # Correcting the path

DATASET_PATHS = {
    "Adult Income": PROCESSED_DIR / "adult_income_processed.csv",
    "Bank Marketing": PROCESSED_DIR / "bank_marketing_processed.csv",
    "Breast Cancer": PROCESSED_DIR / "breast_cancer_processed.csv"
}

missing = []

for name, path in DATASET_PATHS.items():

    if not path.exists():

        missing.append(path.name)

if missing:

    raise FileNotFoundError(

        f"Missing Dataset(s): {missing}"

    )

print("All datasets found.")

All datasets found.


In [20]:
# ============================================================
# Load Processed Datasets
# ============================================================

datasets = {

    name: pd.read_csv(path)

    for name, path in DATASET_PATHS.items()

}

for name, df in datasets.items():

    print(f"{name:20s} {df.shape}")

Adult Income         (48813, 111)
Bank Marketing       (45211, 44)
Breast Cancer        (569, 31)


In [21]:
# ============================================================
# Load Metadata
# ============================================================

metadata = {}

for name, path in METADATA_PATHS.items():

    if path.exists():

        md = SingleTableMetadata.load_from_json(str(path))

    else:

        md = SingleTableMetadata()

        md.detect_from_dataframe(datasets[name])

    metadata[name] = md

print("Metadata Loaded")

Metadata Loaded


In [22]:
# ============================================================
# Dataset Summary
# ============================================================

summary = []

for name, df in datasets.items():

    summary.append({

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing":

        int(df.isna().sum().sum()),

        "Duplicate":

        int(df.duplicated().sum())

    })

summary = pd.DataFrame(summary)

display(summary)

,Dataset,Rows,Columns,Missing,Duplicate
0,Adult Income,48813,111,0,0
1,Bank Marketing,45211,44,0,0
2,Breast Cancer,569,31,0,0


In [23]:
# ============================================================
# Notebook Initialization
# ============================================================

print("="*80)
print("Notebook 04 : Baseline Deep Models")
print("="*80)

print()

print("Completed")

print("✓ Environment")

print("✓ Imports")

print("✓ Configuration")

print("✓ Reproducibility")

print("✓ Paths")

print("✓ Dataset Loading")

print("✓ Metadata Loading")

print()

print("Next Section")

print("▶ Section B : Hyperparameter Definitions")

print("="*80)

gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()

Notebook 04 : Baseline Deep Models

Completed
✓ Environment
✓ Imports
✓ Configuration
✓ Reproducibility
✓ Paths
✓ Dataset Loading
✓ Metadata Loading

Next Section
▶ Section B : Hyperparameter Definitions


In [24]:
# ============================================================
# Global Training Configuration
# ============================================================

GLOBAL_CONFIG = {

    "random_seed": 42,

    "device": DEVICE,

    "num_workers": 2,

    "verbose": True,

    "save_models": True,

    "save_logs": True,

    "save_checkpoints": True,

    "checkpoint_frequency": 25,

    "mixed_precision": False

}

print("Global configuration loaded.")

Global configuration loaded.


In [25]:
# ============================================================
# CTGAN Configuration
# ============================================================

CTGAN_CONFIG = {

    "epochs": 200,

    "batch_size": 128,

    "pac": 1,

    "embedding_dim": 128,

    "generator_dim": (256, 256),

    "discriminator_dim": (256, 256),

    "generator_lr": 2e-4,

    "discriminator_lr": 2e-4,

    "generator_decay": 1e-6,

    "discriminator_decay": 1e-6,

    "discriminator_steps": 1,

    "log_frequency": True,

    "enforce_min_max_values": True,

    "enforce_rounding": True,

    "cuda": torch.cuda.is_available(),

    "verbose": True

}

In [26]:
# ============================================================
# TVAE Configuration
# ============================================================

TVAE_CONFIG = {

    "epochs": 200,

    "batch_size": 128,

    "embedding_dim": 128,

    "compress_dims": (256, 128),

    "decompress_dims": (128, 256),

    "l2scale": 1e-5,

    "loss_factor": 2,

    "cuda": torch.cuda.is_available(),

    "verbose": True

}

In [27]:
# ============================================================
# DP-CTGAN Configuration
# ============================================================

DP_CTGAN_CONFIG = {

    "epochs": 200,

    "batch_size": 128,

    "pac": 1,

    "embedding_dim": 128,

    "generator_dim": (256, 256),

    "discriminator_dim": (256, 256),

    "generator_lr": 2e-4,

    "discriminator_lr": 2e-4,

    "epsilon": 5.0,

    "delta": 1e-5,

    "max_grad_norm": 1.0,

    "noise_multiplier": 1.1,

    "secure_mode": False,

    "cuda": torch.cuda.is_available(),

    "verbose": True

}

In [28]:
# ============================================================
# CTAB-GAN+ Configuration
# ============================================================

CTABGAN_CONFIG = {

    "epochs": 200,

    "batch_size": 128,

    "latent_dim": 128,

    "generator_dim": (256, 256),

    "discriminator_dim": (256, 256),

    "classifier_dim": (128, 128),

    "learning_rate": 2e-4,

    "device": DEVICE

}

In [29]:
# ============================================================
# Dataset-Specific Configuration
# ============================================================

DATASET_CONFIG = {

    "Adult Income": {

        "batch_size": 128

    },

    "Bank Marketing": {

        "batch_size": 128

    },

    "Breast Cancer": {

        "batch_size": 32

    }

}

print("Dataset configuration loaded.")

Dataset configuration loaded.


In [30]:
# ============================================================
# Model Configuration Registry
# ============================================================

MODEL_REGISTRY = {

    "CTGAN": CTGAN_CONFIG,

    "TVAE": TVAE_CONFIG,

    "DP_CTGAN": DP_CTGAN_CONFIG,

    "CTABGAN": CTABGAN_CONFIG

}

print("Available Models:")

for model in MODEL_REGISTRY:
    print(f"✓ {model}")

Available Models:
✓ CTGAN
✓ TVAE
✓ DP_CTGAN
✓ CTABGAN


In [31]:
# ============================================================
# Validate Configurations
# ============================================================

required_models = [

    "CTGAN",

    "TVAE",

    "DP_CTGAN",

    "CTABGAN"

]

for model in required_models:

    assert model in MODEL_REGISTRY, f"{model} configuration missing."

print("All configurations validated successfully.")

All configurations validated successfully.


In [33]:
# ============================================================
# Save Configuration Registry
# ============================================================

config_registry = {

    "global": GLOBAL_CONFIG,

    "ctgan": CTGAN_CONFIG,

    "tvae": TVAE_CONFIG,

    "dp_ctgan": DP_CTGAN_CONFIG,

    "ctabgan": CTABGAN_CONFIG,

    "dataset": DATASET_CONFIG

}

config_path = CONFIG_DIR / "baseline_model_registry.json"

# Ensure the CONFIG_DIR exists before writing the file
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

with open(config_path, "w") as f:
    json.dump(config_registry, f, indent=4, default=str)

print(f"Configuration registry saved to:\n{config_path}")

Configuration registry saved to:
/content/drive/MyDrive/SPP_GAN_Project/configs/baseline_model_registry.json


In [34]:
# ============================================================
# Section Summary
# ============================================================

print("=" * 80)
print("Section B Completed")
print("=" * 80)

print("✓ Global Configuration")
print("✓ CTGAN Configuration")
print("✓ TVAE Configuration")
print("✓ DP-CTGAN Configuration")
print("✓ CTAB-GAN+ Configuration")
print("✓ Dataset Configuration")
print("✓ Model Registry")
print("✓ Configuration Validation")
print("✓ Configuration Export")

print("\nNext Section:")
print("▶ Section C: CTGAN Model Definition and Factory Functions")
print("=" * 80)

Section B Completed
✓ Global Configuration
✓ CTGAN Configuration
✓ TVAE Configuration
✓ DP-CTGAN Configuration
✓ CTAB-GAN+ Configuration
✓ Dataset Configuration
✓ Model Registry
✓ Configuration Validation
✓ Configuration Export

Next Section:
▶ Section C: CTGAN Model Definition and Factory Functions


In [35]:
# ============================================================
# CTGAN Factory Function
# ============================================================

from copy import deepcopy
from sdv.single_table import CTGANSynthesizer


def create_ctgan(metadata, dataset_name=None):
    """
    Create and return a CTGAN synthesizer.

    Parameters
    ----------
    metadata : SingleTableMetadata

    dataset_name : str, optional
        Used to apply dataset-specific overrides.

    Returns
    -------
    CTGANSynthesizer
    """

    config = deepcopy(CTGAN_CONFIG)

    if dataset_name is not None and dataset_name in DATASET_CONFIG:

        for key, value in DATASET_CONFIG[dataset_name].items():

            config[key] = value

    model = CTGANSynthesizer(

        metadata=metadata,

        epochs=config["epochs"],

        batch_size=config["batch_size"],

        pac=config["pac"],

        embedding_dim=config["embedding_dim"],

        generator_dim=config["generator_dim"],

        discriminator_dim=config["discriminator_dim"],

        generator_lr=config["generator_lr"],

        discriminator_lr=config["discriminator_lr"],

        generator_decay=config["generator_decay"],

        discriminator_decay=config["discriminator_decay"],

        discriminator_steps=config["discriminator_steps"],

        enforce_min_max_values=config["enforce_min_max_values"],

        enforce_rounding=config["enforce_rounding"],

        log_frequency=config["log_frequency"],

        verbose=config["verbose"],

        cuda=config["cuda"]

    )

    return model

In [36]:
# ============================================================
# Create CTGAN Models
# ============================================================

ctgan_registry = {}

for dataset_name in metadata.keys():

    ctgan_registry[dataset_name] = create_ctgan(

        metadata=metadata[dataset_name],

        dataset_name=dataset_name

    )

print("="*70)

print("CTGAN Registry")

print("="*70)

for dataset in ctgan_registry:

    print(f"✓ {dataset}")

CTGAN Registry
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer


In [37]:
# ============================================================
# Model Summary
# ============================================================

summary = []

for dataset in ctgan_registry:

    cfg = CTGAN_CONFIG.copy()

    cfg.update(

        DATASET_CONFIG.get(dataset, {})

    )

    summary.append({

        "Dataset": dataset,

        "Epochs": cfg["epochs"],

        "Batch Size": cfg["batch_size"],

        "PAC": cfg["pac"],

        "Embedding": cfg["embedding_dim"],

        "Generator":

        str(cfg["generator_dim"]),

        "Discriminator":

        str(cfg["discriminator_dim"])

    })

summary = pd.DataFrame(summary)

display(summary)

,Dataset,Epochs,Batch Size,PAC,Embedding,Generator,Discriminator
0,Adult Income,200,128,1,128,"(256, 256)","(256, 256)"
1,Bank Marketing,200,128,1,128,"(256, 256)","(256, 256)"
2,Breast Cancer,200,32,1,128,"(256, 256)","(256, 256)"


In [38]:
# ============================================================
# Save CTGAN Configuration
# ============================================================

ctgan_config_file = CONFIG_DIR / "ctgan_config.json"

with open(ctgan_config_file,"w") as f:

    json.dump(

        CTGAN_CONFIG,

        f,

        indent=4,

        default=str

    )

print("CTGAN configuration saved.")

CTGAN configuration saved.


In [39]:
# ============================================================
# Verify Factory
# ============================================================

for dataset_name in datasets:

    model = create_ctgan(

        metadata[dataset_name],

        dataset_name

    )

    assert isinstance(

        model,

        CTGANSynthesizer

    )

print("="*70)

print("CTGAN Factory Verified")

print("="*70)

CTGAN Factory Verified


In [40]:
# ============================================================
# Summary
# ============================================================

print("="*80)

print("Section C Completed")

print("="*80)

print("✓ CTGAN Factory")

print("✓ Dataset-specific Configuration")

print("✓ Model Registry")

print("✓ Configuration Export")

print("✓ Factory Verification")

print()

print("Next Section")

print("▶ Section D : TVAE Model Definition")

print("="*80)

Section C Completed
✓ CTGAN Factory
✓ Dataset-specific Configuration
✓ Model Registry
✓ Configuration Export
✓ Factory Verification

Next Section
▶ Section D : TVAE Model Definition


In [41]:
# ============================================================
# TVAE Factory Function
# ============================================================

from copy import deepcopy
from sdv.single_table import TVAESynthesizer


def create_tvae(metadata, dataset_name=None):
    """
    Create and return a TVAE synthesizer.

    Parameters
    ----------
    metadata : SingleTableMetadata
    dataset_name : str, optional

    Returns
    -------
    TVAESynthesizer
    """

    config = deepcopy(TVAE_CONFIG)

    if dataset_name is not None and dataset_name in DATASET_CONFIG:

        config.update(DATASET_CONFIG[dataset_name])

    model = TVAESynthesizer(

        metadata=metadata,

        epochs=config["epochs"],

        batch_size=config["batch_size"],

        embedding_dim=config["embedding_dim"],

        compress_dims=config["compress_dims"],

        decompress_dims=config["decompress_dims"],

        l2scale=config["l2scale"],

        loss_factor=config["loss_factor"],

        cuda=config["cuda"],

        verbose=config["verbose"]

    )

    return model

In [42]:
# ============================================================
# Create TVAE Registry
# ============================================================

tvae_registry = {}

for dataset_name in metadata.keys():

    tvae_registry[dataset_name] = create_tvae(

        metadata=metadata[dataset_name],

        dataset_name=dataset_name

    )

print("="*70)
print("TVAE Registry")
print("="*70)

for dataset in tvae_registry:

    print(f"✓ {dataset}")

TVAE Registry
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer


In [43]:
# ============================================================
# TVAE Configuration Summary
# ============================================================

summary = []

for dataset in metadata.keys():

    cfg = deepcopy(TVAE_CONFIG)

    cfg.update(DATASET_CONFIG.get(dataset, {}))

    summary.append({

        "Dataset": dataset,

        "Epochs": cfg["epochs"],

        "Batch Size": cfg["batch_size"],

        "Embedding": cfg["embedding_dim"],

        "Encoder": str(cfg["compress_dims"]),

        "Decoder": str(cfg["decompress_dims"]),

        "L2 Scale": cfg["l2scale"],

        "Loss Factor": cfg["loss_factor"]

    })

tvae_summary = pd.DataFrame(summary)

display(tvae_summary)

,Dataset,Epochs,Batch Size,Embedding,Encoder,Decoder,L2 Scale,Loss Factor
0,Adult Income,200,128,128,"(256, 128)","(128, 256)",0.00001,2
1,Bank Marketing,200,128,128,"(256, 128)","(128, 256)",0.00001,2
2,Breast Cancer,200,32,128,"(256, 128)","(128, 256)",0.00001,2


In [44]:
# ============================================================
# Save TVAE Configuration
# ============================================================

config_file = CONFIG_DIR / "tvae_config.json"

with open(config_file, "w") as f:

    json.dump(

        TVAE_CONFIG,

        f,

        indent=4,

        default=str

    )

print(f"Configuration saved:\n{config_file}")

Configuration saved:
/content/drive/MyDrive/SPP_GAN_Project/configs/tvae_config.json


In [45]:
# ============================================================
# Verify TVAE Factory
# ============================================================

for dataset_name in metadata.keys():

    model = create_tvae(

        metadata[dataset_name],

        dataset_name

    )

    assert isinstance(

        model,

        TVAESynthesizer

    )

print("="*70)
print("TVAE Factory Verified Successfully")
print("="*70)

TVAE Factory Verified Successfully


In [46]:
# ============================================================
# Factory Test
# ============================================================

example_model = create_tvae(

    metadata["Adult Income"],

    "Adult Income"

)

print(example_model)

In [47]:
# ============================================================
# Section Summary
# ============================================================

print("="*80)

print("Section D Completed")

print("="*80)

print("✓ TVAE Factory Function")

print("✓ Dataset-specific Configuration")

print("✓ TVAE Registry")

print("✓ Configuration Summary")

print("✓ Configuration Export")

print("✓ Factory Verification")

print()

print("Next Section")

print("▶ Section E : DP-CTGAN Model Definition")

print("="*80)

Section D Completed
✓ TVAE Factory Function
✓ Dataset-specific Configuration
✓ TVAE Registry
✓ Configuration Summary
✓ Configuration Export
✓ Factory Verification

Next Section
▶ Section E : DP-CTGAN Model Definition


In [48]:
# ============================================================
# DP-CTGAN Configuration
# ============================================================

DP_CTGAN_CONFIG = {

    "epochs": 200,

    "batch_size": 128,

    "embedding_dim": 128,

    "generator_dim": (256, 256),

    "discriminator_dim": (256, 256),

    "generator_lr": 2e-4,

    "discriminator_lr": 2e-4,

    "generator_decay": 1e-6,

    "discriminator_decay": 1e-6,

    "discriminator_steps": 1,

    "pac": 1,

    # Differential Privacy

    "epsilon": 5.0,

    "delta": 1e-5,

    "max_grad_norm": 1.0,

    "noise_multiplier": 1.1,

    "secure_mode": False,

    "cuda": torch.cuda.is_available(),

    "verbose": True

}

print("DP-CTGAN configuration loaded.")

DP-CTGAN configuration loaded.


In [49]:
# ============================================================
# DP-CTGAN Wrapper
# ============================================================

from dataclasses import dataclass


@dataclass
class DPCTGANDefinition:
    """
    Configuration container for DP-CTGAN.

    Note:
        The actual Differential Privacy training
        (DP-SGD + gradient clipping + Gaussian noise)
        is implemented in Notebook 06.
    """

    metadata: object

    config: dict

    dataset_name: str

    def summary(self):

        return {

            "Dataset": self.dataset_name,

            "Privacy Budget (ε)": self.config["epsilon"],

            "Delta": self.config["delta"],

            "Noise Multiplier": self.config["noise_multiplier"],

            "Gradient Clip": self.config["max_grad_norm"],

            "Epochs": self.config["epochs"],

            "Batch Size": self.config["batch_size"]

        }

In [50]:
# ============================================================
# DP-CTGAN Factory
# ============================================================

def create_dp_ctgan(metadata, dataset_name):

    config = DP_CTGAN_CONFIG.copy()

    config.update(

        DATASET_CONFIG.get(dataset_name, {})

    )

    return DPCTGANDefinition(

        metadata=metadata,

        config=config,

        dataset_name=dataset_name

    )

In [51]:
# ============================================================
# DP-CTGAN Registry
# ============================================================

dp_ctgan_registry = {

    dataset:

    create_dp_ctgan(

        metadata[dataset],

        dataset

    )

    for dataset in metadata

}

print("DP-CTGAN registry created.")

DP-CTGAN registry created.


In [52]:
# ============================================================
# DP-CTGAN Summary
# ============================================================

summary = pd.DataFrame(

    [

        obj.summary()

        for obj in dp_ctgan_registry.values()

    ]

)

display(summary)

,Dataset,Privacy Budget (ε),Delta,Noise Multiplier,Gradient Clip,Epochs,Batch Size
0,Adult Income,5.0,0.00001,1.1,1.0,200,128
1,Bank Marketing,5.0,0.00001,1.1,1.0,200,128
2,Breast Cancer,5.0,0.00001,1.1,1.0,200,32


In [53]:
# ============================================================
# Save DP Configuration
# ============================================================

config_file = CONFIG_DIR / "dp_ctgan_config.json"

with open(config_file, "w") as f:

    json.dump(

        DP_CTGAN_CONFIG,

        f,

        indent=4,

        default=str

    )

print("Configuration saved.")

Configuration saved.


In [54]:
# ============================================================
# Verification
# ============================================================

assert len(dp_ctgan_registry) == len(metadata)

print("DP-CTGAN configuration verified.")

DP-CTGAN configuration verified.


In [57]:
# ============================================================
# CTAB-GAN+ Configuration
# ============================================================

CTABGAN_CONFIG = {

    # Training

    "epochs": 200,

    "batch_size": 128,

    "random_seed": RANDOM_STATE,

    "device": DEVICE,

    # Network

    "latent_dim": 128,

    "generator_dim": (256, 256),

    "discriminator_dim": (256, 256),

    "classifier_dim": (128, 128),

    # Optimizer

    "learning_rate": 2e-4,

    "beta1": 0.5,

    "beta2": 0.999,

    # Regularization

    "weight_decay": 1e-6,

    "gradient_clip": 1.0,

    # Misc

    "verbose": True

}

print("CTAB-GAN+ configuration loaded.")

CTAB-GAN+ configuration loaded.


In [58]:
# ============================================================
# CTAB-GAN+ Definition
# ============================================================

from dataclasses import dataclass


@dataclass
class CTABGANDefinition:

    metadata: object

    config: dict

    dataset_name: str

    def summary(self):

        return {

            "Dataset": self.dataset_name,

            "Epochs": self.config["epochs"],

            "Batch Size": self.config["batch_size"],

            "Latent Dimension": self.config["latent_dim"],

            "Generator": self.config["generator_dim"],

            "Discriminator": self.config["discriminator_dim"],

            "Learning Rate": self.config["learning_rate"]

        }

In [59]:
# ============================================================
# CTAB-GAN+ Factory
# ============================================================

from copy import deepcopy


def create_ctabgan(metadata, dataset_name):

    config = deepcopy(CTABGAN_CONFIG)

    if dataset_name in DATASET_CONFIG:

        config.update(DATASET_CONFIG[dataset_name])

    return CTABGANDefinition(

        metadata=metadata,

        config=config,

        dataset_name=dataset_name

    )

In [60]:
# ============================================================
# CTAB-GAN+ Registry
# ============================================================

ctabgan_registry = {

    dataset:

    create_ctabgan(

        metadata[dataset],

        dataset

    )

    for dataset in metadata.keys()

}

print("CTAB-GAN+ registry created.")

CTAB-GAN+ registry created.


In [61]:
# ============================================================
# Configuration Summary
# ============================================================

summary = pd.DataFrame(

    [

        model.summary()

        for model in ctabgan_registry.values()

    ]

)

display(summary)

,Dataset,Epochs,Batch Size,Latent Dimension,Generator,Discriminator,Learning Rate
0,Adult Income,200,128,128,"(256, 256)","(256, 256)",0.0002
1,Bank Marketing,200,128,128,"(256, 256)","(256, 256)",0.0002
2,Breast Cancer,200,32,128,"(256, 256)","(256, 256)",0.0002


In [62]:
# ============================================================
# Save Configuration
# ============================================================

config_file = CONFIG_DIR / "ctabgan_config.json"

with open(config_file, "w") as f:

    json.dump(

        CTABGAN_CONFIG,

        f,

        indent=4,

        default=str

    )

print("CTAB-GAN+ configuration saved.")

CTAB-GAN+ configuration saved.


In [63]:
# ============================================================
# Verification
# ============================================================

assert len(ctabgan_registry) == len(metadata)

print("="*70)

print("CTAB-GAN+ Configuration Verified")

print("="*70)

CTAB-GAN+ Configuration Verified


In [64]:
# ============================================================
# Section Summary
# ============================================================

print("="*80)

print("Section F Completed")

print("="*80)

print("✓ CTAB-GAN+ Configuration")

print("✓ Factory Function")

print("✓ Registry")

print("✓ Configuration Export")

print("✓ Verification")

print()

print("Next Section")

print("▶ Section G : Unified Model Registry")

print("="*80)

Section F Completed
✓ CTAB-GAN+ Configuration
✓ Factory Function
✓ Registry
✓ Configuration Export
✓ Verification

Next Section
▶ Section G : Unified Model Registry


In [65]:
# ============================================================
# Unified Model Factory Registry
# ============================================================

MODEL_FACTORY = {

    "CTGAN": create_ctgan,

    "TVAE": create_tvae,

    "DP_CTGAN": create_dp_ctgan,

    "CTABGAN": create_ctabgan

}

print("=" * 70)
print("Available Baseline Models")
print("=" * 70)

for model in MODEL_FACTORY:
    print(f"✓ {model}")

Available Baseline Models
✓ CTGAN
✓ TVAE
✓ DP_CTGAN
✓ CTABGAN


In [66]:
# ============================================================
# Configuration Registry
# ============================================================

MODEL_CONFIG = {

    "CTGAN": CTGAN_CONFIG,

    "TVAE": TVAE_CONFIG,

    "DP_CTGAN": DP_CTGAN_CONFIG,

    "CTABGAN": CTABGAN_CONFIG

}

print("Configuration registry initialized.")

Configuration registry initialized.


In [67]:
# ============================================================
# Create Model Helper
# ============================================================

def create_model(model_name, metadata, dataset_name):
    """
    Generic model creation function.
    """

    if model_name not in MODEL_FACTORY:

        raise ValueError(
            f"Unsupported model: {model_name}"
        )

    return MODEL_FACTORY[model_name](
        metadata,
        dataset_name
    )

In [68]:
# ============================================================
# Configuration Helper
# ============================================================

from copy import deepcopy

def get_model_config(model_name, dataset_name=None):
    """
    Return model configuration merged with
    dataset-specific overrides.
    """

    config = deepcopy(MODEL_CONFIG[model_name])

    if dataset_name in DATASET_CONFIG:

        config.update(
            DATASET_CONFIG[dataset_name]
        )

    return config

In [69]:
# ============================================================
# Validate Factory Functions
# ============================================================

for model_name in MODEL_FACTORY:

    for dataset_name in metadata.keys():

        model = create_model(

            model_name,

            metadata[dataset_name],

            dataset_name

        )

print("Factory validation completed successfully.")

Factory validation completed successfully.


In [70]:
# ============================================================
# Configuration Validation
# ============================================================

required_keys = {

    "CTGAN": ["epochs", "batch_size"],

    "TVAE": ["epochs", "batch_size"],

    "DP_CTGAN": ["epochs", "batch_size", "epsilon"],

    "CTABGAN": ["epochs", "batch_size"]

}

for model_name, keys in required_keys.items():

    config = MODEL_CONFIG[model_name]

    for key in keys:

        assert key in config, f"{key} missing in {model_name}"

print("Configuration validation successful.")

Configuration validation successful.


In [71]:
# ============================================================
#  Export Complete Registry
# ============================================================

registry = {

    "global": GLOBAL_CONFIG,

    "datasets": DATASET_CONFIG,

    "models": MODEL_CONFIG

}

registry_file = CONFIG_DIR / "baseline_registry.json"

with open(registry_file, "w") as f:

    json.dump(

        registry,

        f,

        indent=4,

        default=str

    )

print(f"Registry saved to:\n{registry_file}")

Registry saved to:
/content/drive/MyDrive/SPP_GAN_Project/configs/baseline_registry.json


In [72]:
# ============================================================
# Notebook Verification
# ============================================================

print("=" * 80)
print("Notebook Verification")
print("=" * 80)

print(f"Datasets Loaded      : {len(datasets)}")
print(f"Metadata Objects     : {len(metadata)}")
print(f"Baseline Models      : {len(MODEL_FACTORY)}")

print()

print("Datasets")

for dataset in datasets.keys():

    print(f"✓ {dataset}")

print()

print("Models")

for model in MODEL_FACTORY.keys():

    print(f"✓ {model}")

print()

print("Configuration Registry")

print("✓ Loaded")

print()

print("Notebook 04 Ready")

print("=" * 80)

Notebook Verification
Datasets Loaded      : 3
Metadata Objects     : 3
Baseline Models      : 4

Datasets
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer

Models
✓ CTGAN
✓ TVAE
✓ DP_CTGAN
✓ CTABGAN

Configuration Registry
✓ Loaded

Notebook 04 Ready


In [73]:
# ============================================================
# Cleanup
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print("Memory cleaned.")

Memory cleaned.


In [74]:
# ============================================================
# Final Summary
# ============================================================

print("=" * 90)
print("NOTEBOOK 04 COMPLETED")
print("=" * 90)

print("\nCompleted Sections")

print("✓ Section A : Environment Setup")
print("✓ Section B : Hyperparameter Definitions")
print("✓ Section C : CTGAN Definition")
print("✓ Section D : TVAE Definition")
print("✓ Section E : DP-CTGAN Definition")
print("✓ Section F : CTAB-GAN+ Definition")
print("✓ Section G : Unified Registry")

print("\nOutputs")

print("✓ Model Factory")
print("✓ Configuration Registry")
print("✓ Helper Functions")
print("✓ Validation")
print("✓ JSON Configuration Files")

print("\nNext Notebook")

print("Notebook 05 : Proposed SPP-GAN")

print("\nTraining is intentionally deferred to Notebook 06.")

print("=" * 90)

NOTEBOOK 04 COMPLETED

Completed Sections
✓ Section A : Environment Setup
✓ Section B : Hyperparameter Definitions
✓ Section C : CTGAN Definition
✓ Section D : TVAE Definition
✓ Section E : DP-CTGAN Definition
✓ Section F : CTAB-GAN+ Definition
✓ Section G : Unified Registry

Outputs
✓ Model Factory
✓ Configuration Registry
✓ Helper Functions
✓ Validation
✓ JSON Configuration Files

Next Notebook
Notebook 05 : Proposed SPP-GAN

Training is intentionally deferred to Notebook 06.


In [75]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Project")

for f in PROJECT_ROOT.rglob("*"):
    print(f)

/content/drive/MyDrive/SPP_GAN_Project/config.yaml
/content/drive/MyDrive/SPP_GAN_Project/datasets
/content/drive/MyDrive/SPP_GAN_Project/models
/content/drive/MyDrive/SPP_GAN_Project/synthetic_data
/content/drive/MyDrive/SPP_GAN_Project/results
/content/drive/MyDrive/SPP_GAN_Project/logs
/content/drive/MyDrive/SPP_GAN_Project/reports
/content/drive/MyDrive/SPP_GAN_Project/configs
/content/drive/MyDrive/SPP_GAN_Project/datasets/processed
/content/drive/MyDrive/SPP_GAN_Project/models/baseline_models
/content/drive/MyDrive/SPP_GAN_Project/models/checkpoints
/content/drive/MyDrive/SPP_GAN_Project/synthetic_data/ctgan
/content/drive/MyDrive/SPP_GAN_Project/results/ctgan
/content/drive/MyDrive/SPP_GAN_Project/results/dataset_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/results/dataset_validation.csv
/content/drive/MyDrive/SPP_GAN_Project/results/metadata_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/results/feature_summary.csv
/content/drive/MyDrive/SPP_GAN_Project/logs/ctgan
/co